# Stacking Classifier for Ensemble Brain Metastasis Segmentation

Train a lightweight CNN meta-learner to combine predictions from 4 fine-tuned models.

**Prerequisites:** Run `finetune_ensemble_colab.ipynb` first to generate fine-tuned models.

**Approach:**
- Uses predictions from 4 fine-tuned models (8, 12, 24, 36 patch sizes)
- Extracts disagreement features (variance, range)
- Trains a 3-layer CNN (~4,400 params) to learn optimal combination

**Setup:**
1. Run cells 1-5 in order (setup)
2. Run cell 6 to generate base model predictions
3. Run cell 7 to train stacking classifier
4. Run cell 8 for full-volume evaluation
5. Run cell 9 to compare all methods

**If session disconnects:** Run cells 1-5, then continue from where you left off.

In [ ]:
# 1. Check GPU & Mount Drive
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/brainMetShare'
DRIVE_DIR = '/content/drive/MyDrive/BrainMetShare'
LOCAL_DATA = '/content/data'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model/training_states", exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print("\nDirectories created!")

In [ ]:
# 2. Install Dependencies
!pip install nibabel scipy tqdm monai -q
print("Dependencies installed!")

In [ ]:
# 3. Upload Code
from google.colab import files
import zipfile

print("Upload brainMetShare_code.zip")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('/content/brainMetShare')
        print(f"Extracted {filename}")

# Verify
src_path = '/content/brainMetShare/src/segmentation'
if os.path.exists(src_path):
    print(f"Code ready: {os.listdir(src_path)}")
else:
    print("WARNING: src/segmentation not found - check zip structure")

In [ ]:
# 4. Copy Training Data to Local SSD
import time
import shutil

start = time.time()
!df -h /content

# Create local data directory
os.makedirs(LOCAL_DATA, exist_ok=True)

# Check available data sources
SUPERSET_DIR = f"{DRIVE_DIR}/Superset"
TRAIN_SRC = f"{SUPERSET_DIR}/full/train"
ALT_TRAIN_SRC = f"{DRIVE_DIR}/preprocessed_256/train"

print("Checking data sources...")
print(f"  Superset train: {os.path.exists(TRAIN_SRC)}")
print(f"  Alt train (preprocessed_256): {os.path.exists(ALT_TRAIN_SRC)}")

# Copy train data
LOCAL_TRAIN = f"{LOCAL_DATA}/train"
train_src = TRAIN_SRC if os.path.exists(TRAIN_SRC) else ALT_TRAIN_SRC

if os.path.exists(train_src):
    print(f"\nCopying train data from {train_src}...")
    !cp -r "{train_src}" "{LOCAL_TRAIN}"
    n_train = len([d for d in os.listdir(LOCAL_TRAIN) if os.path.isdir(f"{LOCAL_TRAIN}/{d}")])
    print(f"Train: {n_train} cases")
else:
    print("ERROR: No train data found!")

print(f"\nData copied in {(time.time()-start)/60:.1f} min")
!df -h /content

In [ ]:
# 5. Setup Python Path & Restore Models from Drive
import sys
sys.path.insert(0, '/content/brainMetShare/src')
os.chdir(PROJECT_DIR)

from pathlib import Path

# Paths
MODEL_DIR = Path(f"{PROJECT_DIR}/model")
STATE_DIR = MODEL_DIR / "training_states"
DRIVE_MODEL_DIR = Path(f"{DRIVE_DIR}/model")
DRIVE_STATE_DIR = DRIVE_MODEL_DIR / "training_states"

# Restore fine-tuned models from Drive
print("Restoring fine-tuned models from Drive...")
required_models = [
    'exp1_8patch_finetuned.pth',
    'exp3_12patch_maxfn_finetuned.pth',
    'improved_24patch_finetuned.pth',
    'improved_36patch_finetuned.pth'
]

all_found = True
for model_file in required_models:
    src = DRIVE_MODEL_DIR / model_file
    dst = MODEL_DIR / model_file
    if src.exists():
        shutil.copy(src, dst)
        print(f"  Restored: {model_file}")
    else:
        print(f"  MISSING: {model_file}")
        all_found = False

# Also restore stacking classifier if it exists
stacking_path = DRIVE_MODEL_DIR / 'stacking_classifier.pth'
if stacking_path.exists():
    shutil.copy(stacking_path, MODEL_DIR / 'stacking_classifier.pth')
    print(f"  Restored: stacking_classifier.pth")

if all_found:
    print("\nAll fine-tuned models ready!")
else:
    print("\nWARNING: Some models missing. Run finetune_ensemble_colab.ipynb first.")

In [ ]:
# 6. Generate Base Model Predictions on Validation Set
import gc
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from scipy.ndimage import zoom

from segmentation.unet import LightweightUNet3D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# =============================================================================
# CONFIGURATION
# =============================================================================
ENSEMBLE_CONFIGS = {
    'exp1_8patch': {'patch_size': 8, 'threshold': 0.3},
    'exp3_12patch_maxfn': {'patch_size': 12, 'threshold': 0.25},
    'improved_24patch': {'patch_size': 24, 'threshold': 0.5},
    'improved_36patch': {'patch_size': 36, 'threshold': 0.5},
}

SEQUENCES = ['t1_pre', 't1_gd', 'flair', 't2']
ALT_SEQUENCES = ['t1_pre', 't1_gd', 'flair', 'bravo']
TARGET_SIZE = (128, 128, 128)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def load_volume(case_dir, sequences, target_size=(128, 128, 128)):
    """Load and preprocess a full volume."""
    images = []
    for seq in sequences:
        path = case_dir / f"{seq}.nii.gz"
        if path.exists():
            data = nib.load(str(path)).get_fdata().astype(np.float32)
            factors = [t / s for t, s in zip(target_size, data.shape)]
            data = zoom(data, factors, order=1)
            mean, std = data.mean(), data.std()
            data = (data - mean) / (std + 1e-6)
            images.append(data)
        else:
            images.append(np.zeros(target_size, dtype=np.float32))
    return np.stack(images, axis=0)

def load_mask(case_dir, target_size=(128, 128, 128)):
    """Load ground truth mask."""
    for mask_name in ['seg.nii.gz', 'mask.nii.gz', 'label.nii.gz']:
        path = case_dir / mask_name
        if path.exists():
            data = nib.load(str(path)).get_fdata().astype(np.float32)
            factors = [t / s for t, s in zip(target_size, data.shape)]
            data = zoom(data, factors, order=0)
            return (data > 0.5).astype(np.float32)
    return None

def sliding_window_inference(model, volume, patch_size, overlap=0.5):
    """
    Run inference on full volume using sliding window.
    Returns probability map (not binarized).
    """
    model.eval()
    C, H, W, D = volume.shape
    p = patch_size
    stride = int(p * (1 - overlap))
    
    # Pad volume if needed
    pad_h = (p - H % p) % p if H % stride != 0 else 0
    pad_w = (p - W % p) % p if W % stride != 0 else 0
    pad_d = (p - D % p) % p if D % stride != 0 else 0
    
    orig_shape = (H, W, D)
    
    if pad_h > 0 or pad_w > 0 or pad_d > 0:
        volume = F.pad(torch.from_numpy(volume), (0, pad_d, 0, pad_w, 0, pad_h), mode='constant', value=0).numpy()
        C, H, W, D = volume.shape
    
    output = np.zeros((H, W, D), dtype=np.float32)
    count = np.zeros((H, W, D), dtype=np.float32)
    
    coords = []
    for h in range(0, H - p + 1, stride):
        for w in range(0, W - p + 1, stride):
            for d in range(0, D - p + 1, stride):
                coords.append((h, w, d))
    
    batch_size = 8
    with torch.no_grad():
        for i in range(0, len(coords), batch_size):
            batch_coords = coords[i:i+batch_size]
            patches = []
            
            for h, w, d in batch_coords:
                patch = volume[:, h:h+p, w:w+p, d:d+p]
                patches.append(patch)
            
            batch = torch.from_numpy(np.stack(patches)).float().to(device)
            with torch.amp.autocast('cuda'):
                preds = torch.sigmoid(model(batch)).cpu().numpy()
            
            for j, (h, w, d) in enumerate(batch_coords):
                output[h:h+p, w:w+p, d:d+p] += preds[j, 0]
                count[h:h+p, w:w+p, d:d+p] += 1
    
    output = output / np.maximum(count, 1)
    
    # Remove padding
    H_orig, W_orig, D_orig = orig_shape
    output = output[:H_orig, :W_orig, :D_orig]
    
    return output

# =============================================================================
# LOAD ALL BASE MODELS
# =============================================================================
print("\nLoading base models...")
base_models = {}

for model_name, config in ENSEMBLE_CONFIGS.items():
    model_path = MODEL_DIR / f'{model_name}_finetuned.pth'
    
    model = LightweightUNet3D(
        in_channels=4, out_channels=1,
        base_channels=20, use_attention=True, use_residual=True
    ).to(device)
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    base_models[model_name] = {
        'model': model,
        'patch_size': config['patch_size'],
        'threshold': config['threshold']
    }
    print(f"  Loaded: {model_name}")

# =============================================================================
# GET VALIDATION CASES
# =============================================================================
all_cases = sorted([d for d in Path(LOCAL_TRAIN).iterdir() if d.is_dir()])
n_val = int(len(all_cases) * 0.15)
val_cases = all_cases[-n_val:]

print(f"\nValidation cases: {len(val_cases)}")

# Detect sequences
test_case = val_cases[0]
sequences = SEQUENCES if (test_case / 't2.nii.gz').exists() else ALT_SEQUENCES
print(f"Using sequences: {sequences}")

# =============================================================================
# GENERATE PREDICTIONS FOR ALL VALIDATION CASES
# =============================================================================
print("\nGenerating base model predictions...")
print("This may take a while...")

stacking_data = []

for case_dir in tqdm(val_cases, desc="Processing cases"):
    # Load volume and mask
    volume = load_volume(case_dir, sequences, TARGET_SIZE)
    mask = load_mask(case_dir, TARGET_SIZE)
    
    if mask is None:
        print(f"  Skipping {case_dir.name} - no mask found")
        continue
    
    # Get predictions from all models
    predictions = []
    for model_name in ENSEMBLE_CONFIGS.keys():
        model_info = base_models[model_name]
        pred = sliding_window_inference(
            model_info['model'],
            volume,
            model_info['patch_size'],
            overlap=0.5
        )
        predictions.append(pred)
    
    stacking_data.append({
        'case_name': case_dir.name,
        'predictions': np.stack(predictions, axis=0),  # [4, H, W, D]
        'mask': mask  # [H, W, D]
    })
    
    # Clear GPU memory
    torch.cuda.empty_cache()

print(f"\nGenerated predictions for {len(stacking_data)} cases")

# Save predictions cache to avoid regenerating
print("Saving predictions cache...")
np.savez_compressed(
    MODEL_DIR / 'stacking_predictions_cache.npz',
    data=[{
        'case_name': d['case_name'],
        'predictions': d['predictions'],
        'mask': d['mask']
    } for d in stacking_data]
)
print("Cache saved!")

In [ ]:
# 7. Train Stacking Classifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import numpy as np
from tqdm import tqdm
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# STACKING CLASSIFIER MODEL
# =============================================================================
class StackingClassifier(nn.Module):
    """
    Lightweight 3D CNN meta-learner for combining ensemble predictions.
    
    Input: [B, 6, H, W, D]
        - 4 channels: probability maps from each fine-tuned model
        - 2 channels: disagreement metrics (variance, range)
    
    Output: [B, 1, H, W, D] - combined prediction logits
    
    ~4,400 trainable parameters
    """
    def __init__(self, in_channels=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.Conv3d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.Conv3d(16, 1, kernel_size=1)
        )
    
    def forward(self, x):
        return self.net(x)

# Count parameters
model = StackingClassifier()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stacking Classifier: {n_params:,} trainable parameters")

# =============================================================================
# FEATURE EXTRACTION
# =============================================================================
def extract_stacking_features(predictions):
    """
    Extract features for stacking classifier.
    
    Args:
        predictions: [4, H, W, D] - probability maps from 4 models
    
    Returns:
        features: [6, H, W, D] - model predictions + disagreement metrics
    """
    # Disagreement features
    variance = predictions.var(axis=0, keepdims=True)  # [1, H, W, D]
    range_map = predictions.max(axis=0, keepdims=True) - predictions.min(axis=0, keepdims=True)  # [1, H, W, D]
    
    # Concatenate: [4, H, W, D] + [1, H, W, D] + [1, H, W, D] = [6, H, W, D]
    features = np.concatenate([predictions, variance, range_map], axis=0)
    
    return features.astype(np.float32)

# =============================================================================
# DATASET FOR STACKING
# =============================================================================
class StackingDataset(Dataset):
    """
    Dataset that extracts random 3D patches from stacking features.
    """
    def __init__(self, stacking_data, patch_size=48, patches_per_volume=20):
        self.data = stacking_data
        self.patch_size = patch_size
        self.patches_per_volume = patches_per_volume
    
    def __len__(self):
        return len(self.data) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        item = self.data[vol_idx]
        
        predictions = item['predictions']  # [4, H, W, D]
        mask = item['mask']  # [H, W, D]
        
        # Extract features
        features = extract_stacking_features(predictions)  # [6, H, W, D]
        
        # Random patch
        _, H, W, D = features.shape
        p = self.patch_size
        
        # Bias toward lesion regions (50% of the time)
        if np.random.random() < 0.5 and mask.sum() > 0:
            # Find lesion locations
            lesion_locs = np.where(mask > 0.5)
            if len(lesion_locs[0]) > 0:
                rand_idx = np.random.randint(len(lesion_locs[0]))
                ch = max(0, min(lesion_locs[0][rand_idx] - p//2, H - p))
                cw = max(0, min(lesion_locs[1][rand_idx] - p//2, W - p))
                cd = max(0, min(lesion_locs[2][rand_idx] - p//2, D - p))
            else:
                ch = np.random.randint(0, max(1, H - p))
                cw = np.random.randint(0, max(1, W - p))
                cd = np.random.randint(0, max(1, D - p))
        else:
            ch = np.random.randint(0, max(1, H - p))
            cw = np.random.randint(0, max(1, W - p))
            cd = np.random.randint(0, max(1, D - p))
        
        features_patch = features[:, ch:ch+p, cw:cw+p, cd:cd+p]
        mask_patch = mask[ch:ch+p, cw:cw+p, cd:cd+p]
        
        return (
            torch.from_numpy(features_patch),
            torch.from_numpy(mask_patch[np.newaxis]).float()
        )

# =============================================================================
# LOSS FUNCTIONS (Same as fine-tuning)
# =============================================================================
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        tp = (pred * target).sum()
        fp = (pred * (1 - target)).sum()
        fn = ((1 - pred) * target).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1 - tversky

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.tversky = TverskyLoss(alpha=0.3, beta=0.7)
    
    def focal_loss(self, pred, target, alpha=0.75, gamma=2.0):
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        return (alpha * (1 - pt) ** gamma * bce).mean()
    
    def forward(self, pred, target):
        return 0.7 * self.tversky(pred, target) + 0.3 * self.focal_loss(pred, target)

# =============================================================================
# METRICS
# =============================================================================
def compute_metrics(pred, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred) > threshold).float()
    
    tp = ((pred_binary == 1) & (target == 1)).sum().float()
    tn = ((pred_binary == 0) & (target == 0)).sum().float()
    fp = ((pred_binary == 1) & (target == 0)).sum().float()
    fn = ((pred_binary == 0) & (target == 1)).sum().float()
    
    dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
    sensitivity = (tp + 1e-6) / (tp + fn + 1e-6)
    specificity = (tn + 1e-6) / (tn + fp + 1e-6)
    
    return {
        'dice': dice.item(),
        'sensitivity': sensitivity.item(),
        'specificity': specificity.item()
    }

# =============================================================================
# LOAD CACHED PREDICTIONS (if available)
# =============================================================================
cache_path = MODEL_DIR / 'stacking_predictions_cache.npz'
if cache_path.exists() and 'stacking_data' not in dir():
    print("Loading predictions from cache...")
    cache = np.load(cache_path, allow_pickle=True)
    stacking_data = list(cache['data'])
    print(f"Loaded {len(stacking_data)} cached cases")

# =============================================================================
# TRAINING
# =============================================================================
print("\n" + "="*60)
print("TRAINING STACKING CLASSIFIER")
print("="*60)

# Config
PATCH_SIZE = 48
PATCHES_PER_VOLUME = 20
BATCH_SIZE = 4
EPOCHS = 30
LR = 0.001

# Split data
n_train = int(len(stacking_data) * 0.8)
train_data = stacking_data[:n_train]
val_data = stacking_data[n_train:]

print(f"Train: {len(train_data)} volumes, Val: {len(val_data)} volumes")

# Datasets
train_ds = StackingDataset(train_data, patch_size=PATCH_SIZE, patches_per_volume=PATCHES_PER_VOLUME)
val_ds = StackingDataset(val_data, patch_size=PATCH_SIZE, patches_per_volume=PATCHES_PER_VOLUME)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Model
model = StackingClassifier(in_channels=6).to(device)
criterion = CombinedLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler('cuda')

best_dice = 0
history = {'train_loss': [], 'val_dice': [], 'val_sens': [], 'val_spec': []}

# Training loop
for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0
    n_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for features, masks in pbar:
        features, masks = features.to(device), masks.to(device)
        
        optimizer.zero_grad()
        
        with autocast('cuda'):
            outputs = model(features)
            loss = criterion(outputs, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Validate
    model.eval()
    val_metrics = {'dice': 0, 'sensitivity': 0, 'specificity': 0}
    n_val = 0
    
    with torch.no_grad():
        for features, masks in val_loader:
            features, masks = features.to(device), masks.to(device)
            
            with autocast('cuda'):
                outputs = model(features)
            
            metrics = compute_metrics(outputs, masks, threshold=0.5)
            for k in val_metrics:
                val_metrics[k] += metrics[k]
            n_val += 1
    
    scheduler.step()
    
    # Average metrics
    avg_train_loss = train_loss / n_batches
    for k in val_metrics:
        val_metrics[k] /= n_val
    
    history['train_loss'].append(avg_train_loss)
    history['val_dice'].append(val_metrics['dice'])
    history['val_sens'].append(val_metrics['sensitivity'])
    history['val_spec'].append(val_metrics['specificity'])
    
    print(f"Epoch {epoch}: Loss={avg_train_loss:.4f}, Dice={val_metrics['dice']:.4f}, "
          f"Sens={val_metrics['sensitivity']:.4f}, Spec={val_metrics['specificity']:.4f}")
    
    # Save best
    if val_metrics['dice'] > best_dice:
        best_dice = val_metrics['dice']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'dice': best_dice,
            'sensitivity': val_metrics['sensitivity'],
            'specificity': val_metrics['specificity'],
            'history': history
        }, MODEL_DIR / 'stacking_classifier.pth')
        
        # Also save to Drive
        shutil.copy(MODEL_DIR / 'stacking_classifier.pth', DRIVE_MODEL_DIR / 'stacking_classifier.pth')
        print(f"  Saved best model (Dice={best_dice:.4f})")
    
    if epoch % 10 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print(f"\nTraining complete! Best Dice: {best_dice:.4f}")

In [ ]:
# 8. Full-Volume Evaluation with Stacking
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from scipy.ndimage import zoom
import json

from segmentation.unet import LightweightUNet3D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# STACKING CLASSIFIER (must be defined again if kernel restarted)
# =============================================================================
class StackingClassifier(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.Conv3d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.Conv3d(16, 1, kernel_size=1)
        )
    
    def forward(self, x):
        return self.net(x)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def extract_stacking_features(predictions):
    """Extract features for stacking classifier."""
    variance = predictions.var(axis=0, keepdims=True)
    range_map = predictions.max(axis=0, keepdims=True) - predictions.min(axis=0, keepdims=True)
    features = np.concatenate([predictions, variance, range_map], axis=0)
    return features.astype(np.float32)

def load_volume(case_dir, sequences, target_size=(128, 128, 128)):
    """Load and preprocess a full volume."""
    images = []
    for seq in sequences:
        path = case_dir / f"{seq}.nii.gz"
        if path.exists():
            data = nib.load(str(path)).get_fdata().astype(np.float32)
            factors = [t / s for t, s in zip(target_size, data.shape)]
            data = zoom(data, factors, order=1)
            mean, std = data.mean(), data.std()
            data = (data - mean) / (std + 1e-6)
            images.append(data)
        else:
            images.append(np.zeros(target_size, dtype=np.float32))
    return np.stack(images, axis=0)

def load_mask(case_dir, target_size=(128, 128, 128)):
    """Load ground truth mask."""
    for mask_name in ['seg.nii.gz', 'mask.nii.gz', 'label.nii.gz']:
        path = case_dir / mask_name
        if path.exists():
            data = nib.load(str(path)).get_fdata().astype(np.float32)
            factors = [t / s for t, s in zip(target_size, data.shape)]
            data = zoom(data, factors, order=0)
            return (data > 0.5).astype(np.float32)
    return None

def sliding_window_inference(model, volume, patch_size, overlap=0.5):
    """Run inference on full volume using sliding window."""
    model.eval()
    C, H, W, D = volume.shape
    p = patch_size
    stride = int(p * (1 - overlap))
    
    pad_h = (p - H % p) % p if H % stride != 0 else 0
    pad_w = (p - W % p) % p if W % stride != 0 else 0
    pad_d = (p - D % p) % p if D % stride != 0 else 0
    
    orig_shape = (H, W, D)
    
    if pad_h > 0 or pad_w > 0 or pad_d > 0:
        volume = F.pad(torch.from_numpy(volume), (0, pad_d, 0, pad_w, 0, pad_h), mode='constant', value=0).numpy()
        C, H, W, D = volume.shape
    
    output = np.zeros((H, W, D), dtype=np.float32)
    count = np.zeros((H, W, D), dtype=np.float32)
    
    coords = []
    for h in range(0, H - p + 1, stride):
        for w in range(0, W - p + 1, stride):
            for d in range(0, D - p + 1, stride):
                coords.append((h, w, d))
    
    batch_size = 8
    with torch.no_grad():
        for i in range(0, len(coords), batch_size):
            batch_coords = coords[i:i+batch_size]
            patches = []
            
            for h, w, d in batch_coords:
                patch = volume[:, h:h+p, w:w+p, d:d+p]
                patches.append(patch)
            
            batch = torch.from_numpy(np.stack(patches)).float().to(device)
            with torch.amp.autocast('cuda'):
                preds = torch.sigmoid(model(batch)).cpu().numpy()
            
            for j, (h, w, d) in enumerate(batch_coords):
                output[h:h+p, w:w+p, d:d+p] += preds[j, 0]
                count[h:h+p, w:w+p, d:d+p] += 1
    
    output = output / np.maximum(count, 1)
    
    H_orig, W_orig, D_orig = orig_shape
    output = output[:H_orig, :W_orig, :D_orig]
    
    return output

def compute_volume_metrics(pred, target):
    """Compute metrics on full volume."""
    pred = pred.flatten()
    target = target.flatten()
    
    tp = ((pred == 1) & (target == 1)).sum()
    tn = ((pred == 0) & (target == 0)).sum()
    fp = ((pred == 1) & (target == 0)).sum()
    fn = ((pred == 0) & (target == 1)).sum()
    
    dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
    sensitivity = (tp + 1e-6) / (tp + fn + 1e-6)
    specificity = (tn + 1e-6) / (tn + fp + 1e-6)
    precision = (tp + 1e-6) / (tp + fp + 1e-6)
    
    return {
        'dice': float(dice),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'precision': float(precision)
    }

def count_lesions(mask, min_size=5):
    """Count connected components in binary mask."""
    from scipy.ndimage import label
    labeled, n = label(mask > 0.5)
    # Filter small components
    valid = 0
    for i in range(1, n + 1):
        if (labeled == i).sum() >= min_size:
            valid += 1
    return valid

def lesion_wise_metrics(pred, target, min_size=5, iou_threshold=0.1):
    """Compute lesion-wise detection metrics."""
    from scipy.ndimage import label
    
    pred_labeled, n_pred = label(pred > 0.5)
    target_labeled, n_target = label(target > 0.5)
    
    # Count true positives (detected lesions)
    tp = 0
    for i in range(1, n_target + 1):
        target_lesion = (target_labeled == i)
        if target_lesion.sum() < min_size:
            continue
        
        # Check if any predicted lesion overlaps
        for j in range(1, n_pred + 1):
            pred_lesion = (pred_labeled == j)
            if pred_lesion.sum() < min_size:
                continue
            
            intersection = (target_lesion & pred_lesion).sum()
            union = (target_lesion | pred_lesion).sum()
            iou = intersection / (union + 1e-6)
            
            if iou >= iou_threshold:
                tp += 1
                break
    
    n_gt_valid = sum(1 for i in range(1, n_target + 1) if (target_labeled == i).sum() >= min_size)
    n_pred_valid = sum(1 for j in range(1, n_pred + 1) if (pred_labeled == j).sum() >= min_size)
    
    precision = tp / (n_pred_valid + 1e-6)
    recall = tp / (n_gt_valid + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    
    return {
        'lesion_precision': float(precision),
        'lesion_recall': float(recall),
        'lesion_f1': float(f1),
        'n_lesions_gt': n_gt_valid,
        'n_lesions_pred': n_pred_valid,
        'n_lesions_detected': tp
    }

# =============================================================================
# LOAD MODELS
# =============================================================================
print("Loading models...")

ENSEMBLE_CONFIGS = {
    'exp1_8patch': {'patch_size': 8, 'threshold': 0.3},
    'exp3_12patch_maxfn': {'patch_size': 12, 'threshold': 0.25},
    'improved_24patch': {'patch_size': 24, 'threshold': 0.5},
    'improved_36patch': {'patch_size': 36, 'threshold': 0.5},
}

SEQUENCES = ['t1_pre', 't1_gd', 'flair', 't2']
ALT_SEQUENCES = ['t1_pre', 't1_gd', 'flair', 'bravo']
TARGET_SIZE = (128, 128, 128)

# Load base models
base_models = {}
for model_name, config in ENSEMBLE_CONFIGS.items():
    model_path = MODEL_DIR / f'{model_name}_finetuned.pth'
    
    model = LightweightUNet3D(
        in_channels=4, out_channels=1,
        base_channels=20, use_attention=True, use_residual=True
    ).to(device)
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    base_models[model_name] = {
        'model': model,
        'patch_size': config['patch_size'],
        'threshold': config['threshold']
    }
    print(f"  Loaded: {model_name}")

# Load stacking classifier
stacking_model = StackingClassifier(in_channels=6).to(device)
stacking_ckpt = torch.load(MODEL_DIR / 'stacking_classifier.pth', map_location=device, weights_only=False)
stacking_model.load_state_dict(stacking_ckpt['model_state_dict'])
stacking_model.eval()
print(f"  Loaded: stacking_classifier (Dice={stacking_ckpt['dice']:.4f})")

# =============================================================================
# GET VALIDATION CASES
# =============================================================================
all_cases = sorted([d for d in Path(LOCAL_TRAIN).iterdir() if d.is_dir()])
n_val = int(len(all_cases) * 0.15)
val_cases = all_cases[-n_val:]

test_case = val_cases[0]
sequences = SEQUENCES if (test_case / 't2.nii.gz').exists() else ALT_SEQUENCES

print(f"\nValidation cases: {len(val_cases)}")
print(f"Using sequences: {sequences}")

# =============================================================================
# EVALUATION
# =============================================================================
print("\n" + "="*60)
print("FULL-VOLUME EVALUATION")
print("="*60)

results = {
    'individual_models': {},
    'simple_average': [],
    'max_fusion': [],
    'stacking': []
}

for model_name in ENSEMBLE_CONFIGS.keys():
    results['individual_models'][model_name] = []

for case_dir in tqdm(val_cases, desc="Processing cases"):
    # Load volume and mask
    volume = load_volume(case_dir, sequences, TARGET_SIZE)
    mask = load_mask(case_dir, TARGET_SIZE)
    
    if mask is None:
        continue
    
    # Get predictions from all models
    predictions = []
    for model_name, model_info in base_models.items():
        pred_prob = sliding_window_inference(
            model_info['model'],
            volume,
            model_info['patch_size'],
            overlap=0.5
        )
        predictions.append(pred_prob)
        
        # Individual model metrics
        pred_binary = (pred_prob > model_info['threshold']).astype(np.float32)
        metrics = compute_volume_metrics(pred_binary, mask)
        lesion_metrics = lesion_wise_metrics(pred_binary, mask)
        metrics.update(lesion_metrics)
        results['individual_models'][model_name].append(metrics)
    
    predictions = np.stack(predictions, axis=0)  # [4, H, W, D]
    
    # Simple average ensemble
    avg_pred = predictions.mean(axis=0)
    avg_binary = (avg_pred > 0.5).astype(np.float32)
    metrics = compute_volume_metrics(avg_binary, mask)
    lesion_metrics = lesion_wise_metrics(avg_binary, mask)
    metrics.update(lesion_metrics)
    results['simple_average'].append(metrics)
    
    # Max fusion ensemble
    max_pred = predictions.max(axis=0)
    max_binary = (max_pred > 0.3).astype(np.float32)  # Lower threshold for max
    metrics = compute_volume_metrics(max_binary, mask)
    lesion_metrics = lesion_wise_metrics(max_binary, mask)
    metrics.update(lesion_metrics)
    results['max_fusion'].append(metrics)
    
    # Stacking classifier
    features = extract_stacking_features(predictions)  # [6, H, W, D]
    features_tensor = torch.from_numpy(features[np.newaxis]).float().to(device)  # [1, 6, H, W, D]
    
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            stacking_pred = torch.sigmoid(stacking_model(features_tensor)).cpu().numpy()[0, 0]
    
    stacking_binary = (stacking_pred > 0.5).astype(np.float32)
    metrics = compute_volume_metrics(stacking_binary, mask)
    lesion_metrics = lesion_wise_metrics(stacking_binary, mask)
    metrics.update(lesion_metrics)
    results['stacking'].append(metrics)
    
    torch.cuda.empty_cache()

print("\nEvaluation complete!")

In [ ]:
# 9. Compare All Methods
import numpy as np
import json

print("\n" + "="*80)
print("COMPARISON: Individual Models vs Simple Ensemble vs Stacking")
print("="*80)

def avg_results(results_list):
    """Average metrics across all cases."""
    if not results_list:
        return {}
    keys = results_list[0].keys()
    return {k: np.mean([r[k] for r in results_list]) for k in keys}

def std_results(results_list):
    """Std dev of metrics across all cases."""
    if not results_list:
        return {}
    keys = results_list[0].keys()
    return {k: np.std([r[k] for r in results_list]) for k in keys}

# Aggregate results
summary = {}

# Individual models
print("\n--- Individual Models ---")
print(f"{'Model':<25} {'Dice':<12} {'Sens':<12} {'Spec':<12} {'Lesion F1':<12}")
print("-"*65)

for model_name in ENSEMBLE_CONFIGS.keys():
    avg = avg_results(results['individual_models'][model_name])
    std = std_results(results['individual_models'][model_name])
    summary[model_name] = avg
    print(f"{model_name:<25} {avg['dice']:.4f}+/-{std['dice']:.3f}  "
          f"{avg['sensitivity']:.4f}       {avg['specificity']:.4f}       "
          f"{avg.get('lesion_f1', 0):.4f}")

# Ensemble methods
print("\n--- Ensemble Methods ---")
print(f"{'Method':<25} {'Dice':<12} {'Sens':<12} {'Spec':<12} {'Lesion F1':<12}")
print("-"*65)

for method_name, results_list in [('Simple Average', results['simple_average']),
                                   ('Max Fusion', results['max_fusion']),
                                   ('Stacking Classifier', results['stacking'])]:
    avg = avg_results(results_list)
    std = std_results(results_list)
    summary[method_name] = avg
    print(f"{method_name:<25} {avg['dice']:.4f}+/-{std['dice']:.3f}  "
          f"{avg['sensitivity']:.4f}       {avg['specificity']:.4f}       "
          f"{avg.get('lesion_f1', 0):.4f}")

# Find best method
print("\n--- Best Method Summary ---")
best_dice_method = max(summary.keys(), key=lambda k: summary[k].get('dice', 0))
best_sens_method = max(summary.keys(), key=lambda k: summary[k].get('sensitivity', 0))
best_lesion_method = max(summary.keys(), key=lambda k: summary[k].get('lesion_f1', 0))

print(f"Best Dice:        {best_dice_method} ({summary[best_dice_method]['dice']:.4f})")
print(f"Best Sensitivity: {best_sens_method} ({summary[best_sens_method]['sensitivity']:.4f})")
print(f"Best Lesion F1:   {best_lesion_method} ({summary[best_lesion_method].get('lesion_f1', 0):.4f})")

# Improvement analysis
print("\n--- Stacking vs Others ---")
stacking_dice = summary['Stacking Classifier']['dice']
best_individual = max(summary[k]['dice'] for k in ENSEMBLE_CONFIGS.keys())
simple_avg_dice = summary['Simple Average']['dice']
max_fusion_dice = summary['Max Fusion']['dice']

print(f"Stacking vs Best Individual:  {stacking_dice - best_individual:+.4f} ({(stacking_dice/best_individual - 1)*100:+.1f}%)")
print(f"Stacking vs Simple Average:   {stacking_dice - simple_avg_dice:+.4f} ({(stacking_dice/simple_avg_dice - 1)*100:+.1f}%)")
print(f"Stacking vs Max Fusion:       {stacking_dice - max_fusion_dice:+.4f} ({(stacking_dice/max_fusion_dice - 1)*100:+.1f}%)")

In [ ]:
# 10. Save Results to Drive
import json
import shutil
from datetime import datetime
import matplotlib.pyplot as plt

print("Saving results to Drive...")

# Prepare results for JSON serialization
json_results = {
    'timestamp': datetime.now().isoformat(),
    'n_validation_cases': len(val_cases),
    'summary': {},
    'detailed_results': {}
}

# Summary
for method_name, avg in summary.items():
    json_results['summary'][method_name] = {
        'dice': float(avg.get('dice', 0)),
        'sensitivity': float(avg.get('sensitivity', 0)),
        'specificity': float(avg.get('specificity', 0)),
        'precision': float(avg.get('precision', 0)),
        'lesion_f1': float(avg.get('lesion_f1', 0)),
        'lesion_precision': float(avg.get('lesion_precision', 0)),
        'lesion_recall': float(avg.get('lesion_recall', 0))
    }

# Detailed per-case results
json_results['detailed_results'] = {
    'stacking': results['stacking'],
    'simple_average': results['simple_average'],
    'max_fusion': results['max_fusion']
}

for model_name in ENSEMBLE_CONFIGS.keys():
    json_results['detailed_results'][model_name] = results['individual_models'][model_name]

# Save JSON
results_path = DRIVE_MODEL_DIR / 'stacking_results.json'
with open(results_path, 'w') as f:
    json.dump(json_results, f, indent=2)
print(f"Results saved to: {results_path}")

# Copy stacking model to Drive
if (MODEL_DIR / 'stacking_classifier.pth').exists():
    shutil.copy(MODEL_DIR / 'stacking_classifier.pth', DRIVE_MODEL_DIR / 'stacking_classifier.pth')
    print(f"Model saved to: {DRIVE_MODEL_DIR / 'stacking_classifier.pth'}")

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = list(summary.keys())
metrics = ['dice', 'sensitivity', 'lesion_f1']
titles = ['Dice Coefficient', 'Sensitivity', 'Lesion-wise F1']

for ax, metric, title in zip(axes, metrics, titles):
    values = [summary[m].get(metric, 0) for m in methods]
    colors = ['skyblue'] * len(ENSEMBLE_CONFIGS) + ['lightgreen', 'lightyellow', 'salmon']
    
    bars = ax.bar(range(len(methods)), values, color=colors)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([m.replace('_', '\n') for m in methods], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    
    # Highlight best
    best_idx = np.argmax(values)
    bars[best_idx].set_edgecolor('red')
    bars[best_idx].set_linewidth(2)
    
    # Add value labels
    for i, v in enumerate(values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(DRIVE_MODEL_DIR / 'stacking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nComparison plot saved to: {DRIVE_MODEL_DIR / 'stacking_comparison.png'}")

# Training history plot (if available)
if 'history' in stacking_ckpt:
    history = stacking_ckpt['history']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Stacking Classifier - Training Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['val_dice'], 'g-', label='Dice', linewidth=2)
    ax2.plot(epochs, history['val_sens'], 'b--', label='Sensitivity', alpha=0.7)
    ax2.plot(epochs, history['val_spec'], 'r:', label='Specificity', alpha=0.7)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score')
    ax2.set_title('Stacking Classifier - Validation Metrics')
    ax2.legend()
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(DRIVE_MODEL_DIR / 'stacking_training_curves.png', dpi=150)
    plt.show()
    print(f"Training curves saved to: {DRIVE_MODEL_DIR / 'stacking_training_curves.png'}")

print("\nDone! All results saved to Drive.")